# 💊 Multiple-Pill Recognition and Interaction Safety — Kaggle Master Runner

Notebook này tự động hóa 100% quy trình triển khai ứng dụng **Smart Pill Identifier & Safety Decision Support** trên Kaggle:
- 🚀 **Khối Thị giác (CV):** YOLOv11 Segmentation + ResNet-18 Multi-task + PaddleOCR.
- 📚 **Khối Dữ liệu & RAG:** SQLite CSDL Dược thư + Tra cứu tương tác DDI + Báo cáo lâm sàng.
- 📱 **Giao diện (Frontend):** Streamlit Web & iPhone 17 Pro Max Simulator.
- 🌐 **Public URL:** Tự động mở cổng Internet thông qua **Localtunnel**.

---
## 🔹 BƯỚC 1: Clone Repository & Cài đặt môi trường

In [ ]:
# 1. Xóa thư mục repo cũ nếu có và clone nhánh mergerBe/Fe chuẩn
!rm -rf /kaggle/working/repo
!git clone -b mergerBe/Fe https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git /kaggle/working/repo

# 2. Chuyển vào thư mục repo
%cd /kaggle/working/repo

# 3. Cài đặt các thư viện phụ thuộc
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q streamlit ultralytics paddleocr paddlepaddle-gpu pyngrok
!npm install -g localtunnel

print("✅ BƯỚC 1 HOÀN TẤT: Đã clone mã nguồn và cài đặt toàn bộ package thành công!")

---
## 🔹 BƯỚC 2: Tự động phát hiện & Nạp Models từ 3 Kaggle Datasets

In [ ]:
import os
import shutil
import glob
from pathlib import Path

# Tạo sẵn các thư mục models trong repo
seg_model_dir = Path("models/segmentation_yolov11_full_finetune")
attr_model_dir = Path("models/attribute_resnet18_last_blocks_finetune")
seg_model_dir.mkdir(parents=True, exist_ok=True)
attr_model_dir.mkdir(parents=True, exist_ok=True)

print("🔍 Đang quét và liên kết model artifacts từ /kaggle/input/...")

# 1. Tìm và copy weights YOLOv11 Segmentation
seg_pts = glob.glob("/kaggle/input/pill-segmentation-model/**/*.pt", recursive=True)
if seg_pts:
    target_pt = seg_model_dir / "yolov11m_seg_mediseg_full_finetune_v1.pt"
    shutil.copy(seg_pts[0], target_pt)
    print(f"  ✓ Đã nạp YOLOv11 Segmentation weight: {seg_pts[0]} -> {target_pt}")
else:
    print("  ⚠️ Chưa tìm thấy file .pt trong dataset pill-segmentation-model!")

# 2. Tìm và copy files ResNet18 Attribute (best.pt, label_mapping.json, thresholds, config)
attr_files = glob.glob("/kaggle/input/attrubute-artifact/**/*", recursive=True)
copied_attr = 0
for f in attr_files:
    if os.path.isfile(f):
        dest = attr_model_dir / os.path.basename(f)
        shutil.copy(f, dest)
        copied_attr += 1
print(f"  ✓ Đã nạp {copied_attr} files cấu hình & weights cho ResNet-18 Attribute!")

# 3. Đồng bộ dữ liệu CSDL Dược thư từ dataset (nếu có cập nhật mới)
db_files = glob.glob("/kaggle/input/pill-safety-database/**/*.json", recursive=True)
for f in db_files:
    dest = Path("database_seed") / os.path.basename(f)
    shutil.copy(f, dest)

# 4. Cấu hình file .env sang SQLite nhẹ nhàng cho Kaggle
with open(".env", "w", encoding="utf-8") as f:
    f.write("DATABASE_URL=sqlite:///./medication.db\n")
    f.write("LLM_PROVIDER=fallback\n")

print("✅ BƯỚC 2 HOÀN TẤT: Đã liên kết đầy đủ weights mô hình và cấu hình file .env!")

---
## 🔹 BƯỚC 3: Khởi tạo & Nạp CSDL Dược thư (SQLite Database)

In [ ]:
# Nạp toàn bộ dữ liệu danh mục thuốc, hoạt chất, quy cách ngoại hình, ma trận DDI
!python -m pill_safety.database.scripts.seed

# Kiểm tra dữ liệu sau khi seed
from pill_safety.database.session import SessionLocal
from pill_safety.database.models import DrugProduct, DrugInteraction

with SessionLocal() as db:
    total_drugs = db.query(DrugProduct).count()
    total_ddi = db.query(DrugInteraction).count()
    print(f"  📊 Thống kê CSDL: {total_drugs} sản phẩm thuốc | {total_ddi} cặp tương tác DDI đã sẵn sàng!")

print("✅ BƯỚC 3 HOÀN TẤT: CSDL SQLite đã được nạp đầy đủ 100%!")

---
## 🔹 BƯỚC 4: Lấy Mật khẩu Xác thực Localtunnel (IP của Kaggle)

In [ ]:
import urllib.request

try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=10).read().decode('utf8').strip()
except Exception:
    public_ip = urllib.request.urlopen('https://api.ipify.org', timeout=10).read().decode('utf8').strip()

print("=" * 65)
print(f"🔑 MẬT KHẨU (TUNNEL PASSWORD) CỦA BẠN LÀ:  {public_ip}")
print("=" * 65)
print("👉 Hãy copy dãy số IP ở trên. Khi mở link web, dán IP này vào ô 'Tunnel Password' rồi bấm Submit!")

---
## 🔹 BƯỚC 5: Khởi chạy Streamlit Web Server & Tạo Public URL

In [ ]:
import subprocess
import time

# 1. Dọn dẹp process cổng 8501 cũ nếu có
!fuser -k 8501/tcp 2>/dev/null || true

# 2. Khởi chạy Streamlit ngầm ở cổng 8501
cmd_streamlit = "streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false"
streamlit_process = subprocess.Popen(cmd_streamlit, shell=True)

time.sleep(3)
print("🚀 Streamlit Server đang hoạt động ngầm trên cổng 8501...")
print("🌐 Đang kết nối Localtunnel để lấy đường link truy cập công khai...")

# 3. Khởi chạy Localtunnel
!npx localtunnel --port 8501